In [2]:
import pandas as pd
from pathlib import Path

import pandas as pd
from pathlib import Path

def _read_evolving_csv(csv_path: Path) -> pd.DataFrame:
    """
    Liest die experiment_summary.csv auch dann fehlerfrei ein, wenn neuere Runs 
    zusätzliche Spalten (z.B. mean_c_at_best_loss, mean_c_last_epoch) haben als ältere Runs.
    """
    try:
        return pd.read_csv(csv_path)
    except pd.errors.ParserError:
        import csv
        with open(csv_path, "r") as f:
            rows = list(csv.reader(f))
        header = rows[0]
        max_cols = max(len(r) for r in rows)
        if max_cols > len(header):
            # Falls der Header noch aus dem alten Logger stammt, die 4 neuen Spalten an der richtigen Stelle einfügen
            idx = header.index("fold_c_indices") + 1
            extra_cols = ["mean_c_at_best_loss", "std_c_at_best_loss", "mean_c_last_epoch", "std_c_last_epoch"]
            header = header[:idx] + extra_cols[:max_cols - len(header)] + header[idx:]
        
        padded_rows = [
            r[:idx] + [None] * (max_cols - len(r)) + r[idx:] if len(r) < max_cols else r
            for r in rows[1:]
        ]
        df = pd.DataFrame(padded_rows, columns=header)
        for col in df.columns:
            if col.startswith(("mean_", "std_", "lr", "dropout", "loss_alpha", "weight_decay")):
                df[col] = pd.to_numeric(df[col], errors="coerce")
        return df


def create_summary_table(
    csv_path: Path, 
    decimals: int = 3, 
    metric: str = "all"  # Optionen: "all", "best_c", "best_loss", "last_epoch"
) -> pd.DataFrame:
    df = _read_evolving_csv(csv_path)
    
    # Jeweils neuesten Run pro (cohort, model_type) behalten
    df = df.drop_duplicates(subset=["cohort", "model_type"], keep="last").copy()
    
    # 1. Kohorten-Namen kürzen ("TCGA-KIRC" -> "KIRC")
    df["Dataset"] = df["cohort"].str.replace("TCGA-", "", regex=False)
    
    # 2. Modellnamen auf Paper-Bezeichnungen mappen
    model_names = {
        "abmil": "Attention MIL (WSI Only)",
        "snn": "SNN (Genomic Only)",
        "deepsets": "Deep Sets (WSI Only)",
        "mcat": "MCAT (Multimodal)"
    }
    df["BaseModel"] = df["model_type"].map(lambda x: model_names.get(x, str(x).upper()))
    
    # 3. Verfügbare Metrik-Spalten definieren
    metric_defs = {
        "best_c": ("mean_c_index", "std_c_index", "Best Val C-Index"),
        "best_loss": ("mean_c_at_best_loss", "std_c_at_best_loss", "@ Best Val Loss"),
        "last_epoch": ("mean_c_last_epoch", "std_c_last_epoch", "Last Epoch")
    }
    
    selected_keys = list(metric_defs.keys()) if metric == "all" else [metric]
    records = []
    
    for key in selected_keys:
        mean_col, std_col, label = metric_defs[key]
        if mean_col not in df.columns or df[mean_col].isna().all():
            continue
            
        sub = df.dropna(subset=[mean_col]).copy()
        sub["Model"] = sub["BaseModel"] if metric != "all" else sub["BaseModel"] + f" [{label}]"
        sub["mean_val"] = sub[mean_col].astype(float)
        sub["std_val"] = sub[std_col].astype(float)
        sub["Result"] = sub.apply(
            lambda r: f"{r['mean_val']:.{decimals}f} ± {r['std_val']:.{decimals}f}",
            axis=1
        )
        records.append(sub[["Model", "Dataset", "Result", "mean_val"]])
        
    full_df = pd.concat(records, ignore_index=True)
    
    # 4. Pivot-Tabelle erzeugen (Reihenfolge der Zeilen beibehalten)
    row_order = full_df["Model"].drop_duplicates().tolist()
    table = full_df.pivot(index="Model", columns="Dataset", values="Result").reindex(row_order)
    
    # Spalten in der MCAT-Reihenfolge sortieren
    col_order = [c for c in ["KIRC", "GBMLGG", "KIRP"] if c in table.columns]
    remaining_cols = [c for c in table.columns if c not in col_order]
    table = table[col_order + remaining_cols]
    
    # 5. Overall Mean über alle Datensätze pro Zeile berechnen
    overall_mean = full_df.groupby("Model", sort=False)["mean_val"].mean()
    table["Overall"] = overall_mean.map(lambda x: f"{x:.{decimals}f}")
    
    table.columns.name = None
    return table.reset_index()


csv_file = Path("./runs/experiment_summary.csv")

# Zeigt alle 3 Auswertungsarten (Best C-Index, @ Best Loss, Last Epoch) im direkten Vergleich:
summary_table = create_summary_table(csv_file, decimals=3, metric="all")
display(summary_table.style.hide(axis="index"))

Model,KIRC,GBMLGG,KIRP,Overall
Attention MIL (WSI Only) [Best Val C-Index],0.738 ± 0.074,0.767 ± 0.068,0.770 ± 0.106,0.758
Attention MIL (WSI Only) [@ Best Val Loss],0.727 ± 0.072,0.758 ± 0.072,0.744 ± 0.117,0.743
Attention MIL (WSI Only) [Last Epoch],0.697 ± 0.088,0.750 ± 0.071,0.757 ± 0.105,0.735
